# HW4: Walking Patterns — Where and When to Step
**NTUT MuJoCo Course — Lecture 5**

Lecture 4 stabilized one equilibrium. Walking keeps changing which foot supports the body.
We will plan contact timing, understand a simplified center-of-mass model, then inspect
Open Duck Mini V2 trajectories produced by Placo.

| Part | Points | Topic |
|---|---|---|
| A | 5 | Support phases and step commands |
| B | 10 | LIPM, ZMP and support polygons |
| C | 10 | Smooth swing trajectories and Open Duck analysis |
| D | +5 bonus | Height/timing parameter study |

### Rules
- **DO NOT USE `while` LOOPS.** Use bounded `for` loops.
- **DO NOT ALTER TEST CELLs**, fixed parameters or tolerances.
- Every student function initially raises `NotImplementedError`.
- Placo and robot-loading code are given. Your formulas are used in the plots and diagnostics.
- See `README.md` for the **Python (walking-course)** kernel. Each lecture runs independently.

In [ ]:
%matplotlib inline
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from scipy.linalg import expm
from scipy.spatial.transform import Rotation
from IPython.display import display

for root in (Path.cwd(), *Path.cwd().parents):
    if (root / 'lecture5' / 'openduck').is_dir():
        COURSE_ROOT = root
        sys.path.insert(0, str(root / 'lecture5'))
        break
else:
    raise FileNotFoundError('Run this notebook from inside the course repository.')
from openduck import (WalkConfig, generate_reference, support_halfspaces, diagnostics,
                      print_diagnostics, validate_episode, save_episode, load_episode, frame_slices)
from openduck.visualization import plot_reference, plot_joint_motion
np.set_printoptions(precision=6, suppress=True)
print('imports ok')

## From a fixed point to a repeating step
A **stance** foot supports the robot; a **swing** foot moves to its next placement.
During **single support**, one foot is scheduled to touch the floor; during **double
support**, both are. A hybrid model combines continuous motion and discrete contact events.

A rimless wheel is a useful small example. Between collisions,
$\ddot\theta=(g/l)\sin\theta$. At a forward foot strike,
$\dot\theta^+=\cos(2\alpha)\dot\theta^-$ and the stance coordinate resets from
$\gamma+\alpha$ to $\gamma-\alpha$. On the forward-stepping branch:
$$v_{n+1}=\cos(2\alpha)\sqrt{v_n^2+4(g/l)\sin\alpha\sin\gamma}.$$
The following **given** demo plots the step-to-step map; it does not simulate impacts in
Open Duck. A periodic gait is not a fixed pose, and a repeating reference is not proof of
stable walking. Full compass-gait equations and impact proofs are outside this homework.

Reading: [MIT: simple models of walking](https://underactuated.csail.mit.edu/simple_legs.html#section2).

In [ ]:
# Given demo: use parameters/initial speed on the forward-stepping branch.
alpha, gamma, g, leg_length = np.pi/8, 0.15, 9.81, 1.0
threshold = np.sqrt(2*g/leg_length*(1-np.cos(gamma-alpha)))
speeds = [1.4]
for _ in range(12):
    assert speeds[-1] > threshold
    speeds.append(np.cos(2*alpha)*np.sqrt(speeds[-1]**2 + 4*g/leg_length*np.sin(alpha)*np.sin(gamma)))
plt.plot(speeds, 'o-')
plt.xlabel('Step number'); plt.ylabel('Post-impact angular speed (rad/s)')
plt.title('A repeating gait appears as a fixed point of the return map')
plt.grid(alpha=0.3); plt.show()

## Part A (5 pts): Contact timing and commands
For this **teaching schedule**, start with left single support. Use half-open intervals:

| Interval within a cycle | Contact label [left, right] |
|---|---|
| $[0,T_s)$ | [1,0] |
| $[T_s,T_s+T_d)$ | [1,1] |
| $[T_s+T_d,2T_s+T_d)$ | [0,1] |
| $[2T_s+T_d,2(T_s+T_d))$ | [1,1] |

The cycle period is $T=2(T_s+T_d)$. Apply modulo to handle repeated cycles; allow
$T_d=0$. Raise `ValueError` for negative time, nonpositive $T_s$, or negative $T_d$.
Placo has its own start/end phases, so compare the pattern rather than assuming identical
phase origins. `dx`, `dy`, `dtheta` are **per-step displacements**, not velocities.
For two equal steps per cycle the nominal rate is $2\,\Delta/T$.

In [ ]:
def support_phase(t, single_support, double_support):
    """Return a length-2 integer contact array for a scalar time in seconds."""
    if not np.isfinite([t, single_support, double_support]).all() or t < 0 or single_support <= 0 or double_support < 0:
        raise ValueError('Invalid phase parameters')
    # TODO: implement the formula described above.
    raise NotImplementedError("support_phase not implemented")

In [ ]:
def step_rate(displacement, period):
    """Return displacement per second; period must be positive."""
    if not np.isfinite(period) or period <= 0:
        raise ValueError('period must be positive')
    # TODO: implement the formula described above.
    raise NotImplementedError("step_rate not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A)
# Binary-exact times make the boundary convention unambiguous.
s, d = 0.25, 0.125
for t, expected in [(0,[1,0]),(s-1e-8,[1,0]),(s,[1,1]),(s+d,[0,1]),
                    (2*s+d,[1,1]),(0.75,[1,0]),(1.125,[0,1])]:
    np.testing.assert_array_equal(support_phase(t,s,d), expected)
np.testing.assert_array_equal(support_phase(0.25,0.25,0),[0,1])
np.testing.assert_allclose(step_rate([0.02,0.01,0.05],0.4),[0.1,0.05,0.25])
for args in [(-1,s,d),(0,0,d),(0,s,-1)]:
    try: support_phase(*args)
    except ValueError: pass
    else: raise AssertionError('invalid support parameters accepted')
print('Part A tests passed.')

## Part B (10 pts): From CoM acceleration to ZMP
For flat ground, constant CoM height $h>0$, negligible change of angular momentum about
CoM, and maintained contact, each horizontal axis obeys the linear inverted pendulum model:
$$\ddot c=\omega^2(c-p),\qquad \omega=\sqrt{g/h},\qquad p=c-\ddot c/\omega^2.$$
Here $c$ is horizontal center-of-mass position and $p$ is ZMP/CoP on the ground. Under these
assumptions, it must lie inside the active support polygon. CoM projection itself need not
remain inside that polygon during dynamic walking. ZMP containment alone is not full
physical feasibility.

For **constant** $p$ over $\Delta t$, the exact update is:
$$c^+=p+\cosh(\omega\Delta t)(c-p)+\frac{\sinh(\omega\Delta t)}{\omega}\dot c,$$
$$\dot c^+=\omega\sinh(\omega\Delta t)(c-p)+\cosh(\omega\Delta t)\dot c.$$
Implement for scalar or array-valued $c,\dot c,p$. Return `(position, velocity)`.
Then reconstruct ZMP and test polygon halfspaces $Ap\le b$ (including its boundary).
`support_halfspaces` is given; its outward normals have unit length.

Reading: [MIT: CoP, ZMP and planning](https://underactuated.csail.mit.edu/humanoids.html).

In [ ]:
def lipm_step(position, velocity, zmp, height, dt, gravity=9.81):
    """Exact zero-order-hold LIPM step; require height, gravity > 0 and dt >= 0."""
    if height <= 0 or gravity <= 0 or dt < 0:
        raise ValueError('Invalid LIPM parameters')
    # TODO: implement the formula described above.
    raise NotImplementedError("lipm_step not implemented")

In [ ]:
def reconstruct_zmp(position, acceleration, height, gravity=9.81):
    """Return horizontal ZMP with the same shape as position."""
    if height <= 0 or gravity <= 0:
        raise ValueError('height and gravity must be positive')
    # TODO: implement the formula described above.
    raise NotImplementedError("reconstruct_zmp not implemented")

In [ ]:
def inside_support(point, polygon, tolerance=1e-9):
    """Return bool; boundary is included. polygon is an array of world xy vertices."""
    # TODO: implement the formula described above.
    raise NotImplementedError("inside_support not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B)
# Independent oracle: exponential of the affine dynamics in augmented coordinates.
rng = np.random.default_rng(5)
for height in (0.2,0.5):
    for dt in (0,0.01,0.1):
        c,v,p = rng.normal(size=3)
        M = np.array([[0,1,0],[9.81/height,0,-9.81/height*p],[0,0,0]])
        expected = (expm(M*dt)@np.array([c,v,1]))[:2]
        np.testing.assert_allclose(lipm_step(c,v,p,height,dt),expected,atol=1e-10)
np.testing.assert_allclose(lipm_step([0.1,0.2],[0,0],[0.1,0.2],0.2,0.1),[[0.1,0.2],[0,0]])
np.testing.assert_allclose(reconstruct_zmp([0.1,0.2],[4.905,0],0.2),[0,0.2])
square=np.array([[-1,-1],[1,-1],[1,1],[-1,1]])
assert inside_support([1,0],square) and not inside_support([1.01,0],square)
angle=0.6; R2=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]])
rotated=square@R2.T+[3,4]
assert inside_support([3,4],rotated) and not inside_support([6,4],rotated)
for args in [(0,0,0,0,0.01),(0,0,0,0.2,-0.01)]:
    try: lipm_step(*args)
    except ValueError: pass
    else: raise AssertionError('invalid LIPM parameters accepted')
print('Part B tests passed.')

### Why future footsteps matter
The LIPM has an unstable mode: an arbitrary CoM initial condition will diverge even when
ZMP stays fixed inside a foot. The next plot deliberately demonstrates that failure.
Placo plans a whole CoM trajectory across future supports, rather than just integrating
an arbitrary initial condition. We do not implement its full QP in this assignment.

In [ ]:
positions, velocities = [0.01], [0.0]
for _ in range(100):
    c,v=lipm_step(positions[-1],velocities[-1],0.0,0.2,0.01)
    positions.append(float(c)); velocities.append(float(v))
plt.plot(np.arange(101)*0.01,positions)
plt.xlabel('Time (s)'); plt.ylabel('CoM x (m)'); plt.title('Fixed ZMP does not stabilize an arbitrary CoM state')
plt.grid(alpha=0.3); plt.show()

## Part C (10 pts): Swing trajectory and Open Duck
For normalized swing time $u=t/T\in[0,1]$, use
$$s(u)=3u^2-2u^3,\qquad p(u)=(1-s)p_0+s p_1+[0,0,16Hu^2(1-u)^2]^T.$$
This has zero endpoint velocity and adds height $H$ at mid-swing. For unequal endpoint
heights, $H$ is measured above their interpolated midpoint. This is a **teaching curve**;
Placo's own swing construction may differ. Return position and velocity using the
analytic derivatives with respect to time. Reject time outside the swing interval.

In [ ]:
def swing_foot(t, duration, start, end, height):
    """Return (position, velocity), each a 3-vector, for scalar time."""
    if duration <= 0 or height < 0 or t < 0 or t > duration:
        raise ValueError('Invalid swing parameters')
    # TODO: implement the formula described above.
    raise NotImplementedError("swing_foot not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (C1)
a,b=np.array([0,0,0]),np.array([0.1,0.02,0])
for t,p in [(0,a),(0.4,b)]:
    pos,vel=swing_foot(t,0.4,a,b,0.04)
    np.testing.assert_allclose(pos,p,atol=1e-12)
    np.testing.assert_allclose(vel,0,atol=1e-12)
np.testing.assert_allclose(swing_foot(0.2,0.4,a,b,0.04)[0],[0.05,0.01,0.04])
for t in (0.03,0.11,0.29):
    eps=1e-6
    numeric=(swing_foot(t+eps,0.4,a,b,0.04)[0]-swing_foot(t-eps,0.4,a,b,0.04)[0])/(2*eps)
    np.testing.assert_allclose(swing_foot(t,0.4,a,b,0.04)[1],numeric,atol=1e-8)
try: swing_foot(-0.01,0.4,a,b,0.04)
except ValueError: pass
else: raise AssertionError('negative swing time accepted')
print('Part C1 tests passed.')
t_swing=np.linspace(0,0.4,101)
p_swing=np.array([swing_foot(t,0.4,a,b,0.04)[0] for t in t_swing])
plt.plot(t_swing,p_swing); plt.legend(['x','y','z'])
plt.xlabel('Swing time (s)'); plt.ylabel('Position (m)'); plt.grid(alpha=0.3); plt.show()

### Given implementation: inspect the real planning pipeline
`generate_reference` wraps the adapted Open Duck engine. It creates repetitive footsteps,
builds supports, calls `WalkPatternGenerator`, and uses `WalkTasks` / `KinematicsSolver`.
The heavy planning and IK are given; Lecture 6 explains the IK and recording stages.

`planned_com` and `planned_acceleration` describe the desired trajectory; `com` and
`foot_T` are computed by FK from generated joint states. They are **not physics measurements**.
`contacts` are scheduled support labels. All transforms here are in world coordinates.
The initial 2 s are omitted; the 6 s recording contains 300 samples at 50 Hz.

In the exploratory cell, compare `WalkConfig(dx=0.01)`,
`WalkConfig(foot_height=0.025)`, and `WalkConfig(single_support=0.22)` one at a time.
Predict which plot changes first, observe the result, and restore `WalkConfig()` for submission.
The support polygons are the planner's geometric model, not measured sole contact patches.


The source intentionally disables joint-position limits. We retain its reference behavior
and visibly report URDF violations instead of changing the limits to make tests pass.
The source planner does not ensure ZMP containment at every exported sample. Its discrete planning constraints must be checked at the output times too; report violations instead of assuming success.
A moving animation, small ZMP residual, or good IK tracking does not establish hardware feasibility.

In [ ]:
# Given integration. Change COMMAND only in this exploratory cell, not in tests.
COMMAND=WalkConfig()
samples=generate_reference(COMMAND)
print('samples:',len(samples['time']),'joints:',len(samples['joint_names']))
print('nominal [vx, vy, yaw rate]:',step_rate([COMMAND.dx,COMMAND.dy,COMMAND.dtheta],samples['period']))
report=print_diagnostics(samples)
plot_reference(samples); plt.show()
# Your Part B functions now check a real planned trajectory.
zmp_from_formula=reconstruct_zmp(samples['planned_com'][:,:2],samples['planned_acceleration'][:,:2],COMMAND.com_height)
inside=np.array([inside_support(p,poly) for p,poly in zip(zmp_from_formula,samples['polygons'])])
print('ZMP formula max error:',np.max(np.abs(zmp_from_formula-samples['zmp'])))
print('ZMP samples inside active polygon:',inside.mean())

In [ ]:
# ------------------------------------------------------------------- TEST CELL (C2)
assert samples['joints'].shape==(round(COMMAND.duration*COMMAND.fps),16)
np.testing.assert_allclose(np.diff(samples['engine_time']),1/COMMAND.fps,atol=1e-10)
np.testing.assert_allclose(zmp_from_formula,samples['zmp'],atol=1e-8)
assert int((~inside).sum()) == report['zmp_outside_samples'], 'containment checks disagree'
if not inside.all():
    print('CONTAINMENT VIOLATION:',int((~inside).sum()),'samples; this is not a fully feasible trajectory.')
assert np.isfinite(samples['joints']).all()
assert set(map(tuple,samples['contacts'])).issubset({(1,0),(0,1),(1,1)})
assert report['max_com_error_m'] < 0.01 and report['max_foot_error_m'] < 0.01
assert report['physics_verified'] is False
print('Part C2 tests passed (kinematic checks only; inspect limit violations above).')

### Short answers (included in Part C)
1. Why can a moving robot's CoM projection leave a support polygon while its ZMP remains inside?
2. Why is a scheduled contact label not evidence of a measured ground force?
3. Compare the teaching swing curve with the generated foot-height plot. Which boundary properties agree?
4. What do reported URDF joint-limit or sampled ZMP-containment violations prevent us from claiming?

**PUT YOUR ANSWERS HERE**

## Part D (+5 bonus): One parameter at a time
Generate a short comparison at baseline, CoM height 0.21 m, and single-support duration
0.22 s. Keep the step command unchanged. Return a list of dictionaries with `height`,
`single_support`, `omega`, `nominal_vx`, `min_zmp_margin_m`, `max_foot_error_m` and
`joint_violations_rad`. Use your `step_rate`; diagnostics and generation are given.
A smaller $\omega=\sqrt{g/h}$ does not by itself prove that walking is more stable.

In [ ]:
def parameter_study(base):
    """Generate three 2-second cases and return the metrics described above."""
    # TODO: implement the formula described above.
    raise NotImplementedError("parameter_study not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (D, BONUS)
rows=parameter_study(WalkConfig())
assert len(rows)==3
assert rows[1]['omega']<rows[0]['omega']
assert rows[2]['nominal_vx']<rows[0]['nominal_vx']
for row in rows:
    assert np.isfinite([row['min_zmp_margin_m'],row['max_foot_error_m']]).all()
    print(row)
print('Part D tests passed.')

Explain which change reduced the nominal speed and which changed the LIPM growth rate.
Describe the observed support margin and joint-limit changes without calling either a stability proof.

**PUT YOUR ANSWER HERE**

### Submitting
1. Fill every TODO and answer the short questions. Do not change TEST CELLs.
2. **Kernel → Restart Kernel and Run All Cells**. All required tests must pass; bonus is optional.
3. Export to HTML and print to PDF. Submit the PDF and your `.ipynb`.
4. Lecture 6 also submits the generated JSON. No GUI is needed for grading.

The model is a kinematic reference generator. Passing these tests is not a certificate of
physical balance, friction feasibility, collision freedom, or torque feasibility.